# Fases 2, 3 e 4 - Exploração

## Fase 2: Coleta de Séries Históricas de Preço
## Fase 3: Análise de Séries de Arbitragem
## Fase 4: Estatísticas de Spread

Este notebook explora a coleta e análise de séries de preço YES/NO.

In [ ]:
# Setup do path para imports
import sys
sys.path.insert(0, '..')

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from config.settings import (
    DATA_RAW_DIR,
    DATA_PROCESSED_DIR,
    DEFAULT_TIMEFRAME,
    ARBITRAGE_THRESHOLD,
    ensure_data_dirs_exist,
)
from core.api_client import get_api_client
from core.models import Market
from pipeline.phase1_market_selection import load_markets_from_json
from pipeline.phase2_price_history import (
    download_market_price_history,
    save_price_history_to_csv,
    load_price_history_from_csv,
    get_price_history_stats,
)

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)

ensure_data_dirs_exist()

## 1. Carregar Mercados Selecionados

In [ ]:
# Carrega mercados do JSON gerado na Fase 1
try:
    groups = load_markets_from_json()
    print(f"Grupo A: {len(groups['A'])} mercados")
    print(f"Grupo B: {len(groups['B'])} mercados")
    print(f"Grupo C: {len(groups['C'])} mercados")
except FileNotFoundError:
    print("Arquivo de mercados não encontrado. Execute a Fase 1 primeiro.")
    groups = {'A': [], 'B': [], 'C': []}

## 2. Fase 2: Coleta de Histórico de Preços

In [ ]:
# Seleciona um mercado de exemplo do Grupo A para análise
if groups['A']:
    sample_market = groups['A'][0]
    print(f"Mercado de exemplo:")
    print(f"  ID: {sample_market.id}")
    print(f"  Pergunta: {sample_market.question[:80]}...")
    print(f"  Volume: ${sample_market.volume:,.0f}")
    print(f"  YES Token: {sample_market.yes_token_id}")
    print(f"  NO Token: {sample_market.no_token_id}")

In [ ]:
# Baixa histórico de preços para o mercado de exemplo
if groups['A']:
    api_client = get_api_client()
    
    price_history = download_market_price_history(
        sample_market,
        timeframe=DEFAULT_TIMEFRAME,
        api_client=api_client,
    )
    
    if price_history and price_history.has_data:
        filepath = save_price_history_to_csv(price_history)
        print(f"Histórico salvo em: {filepath}")
        print(f"Pontos YES: {len(price_history.yes_prices)}")
        print(f"Pontos NO: {len(price_history.no_prices)}")

## 3. Análise de Série de Preços

In [ ]:
# Carrega histórico do CSV
if groups['A']:
    df = load_price_history_from_csv(sample_market.id, DEFAULT_TIMEFRAME)
    
    if df is not None:
        print("Primeiras linhas do histórico:")
        display(df.head(10))
        
        print("\nEstatísticas:")
        stats = get_price_history_stats(df)
        for key, value in stats.items():
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in value.items():
                    print(f"    {k}: {v}")
            else:
                print(f"  {key}: {value}")

In [ ]:
# Visualização da série de preços
if groups['A'] and df is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Preços YES e NO
    axes[0].plot(df['timestamp'], df['price_yes'], label='YES', color='green', alpha=0.8)
    axes[0].plot(df['timestamp'], df['price_no'], label='NO', color='red', alpha=0.8)
    axes[0].set_ylabel('Preço')
    axes[0].set_title(f'Preços YES/NO - {sample_market.question[:50]}...')
    axes[0].legend()
    axes[0].set_ylim(0, 1)
    
    # Spread (1 - YES - NO)
    axes[1].plot(df['timestamp'], df['spread'], label='Spread', color='blue', alpha=0.8)
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[1].fill_between(df['timestamp'], df['spread'], 0, 
                         where=(df['spread'] > 0), alpha=0.3, color='green', label='Arbitragem')
    axes[1].fill_between(df['timestamp'], df['spread'], 0,
                         where=(df['spread'] < 0), alpha=0.3, color='red', label='Overpriced')
    axes[1].set_xlabel('Data')
    axes[1].set_ylabel('Spread (1 - YES - NO)')
    axes[1].set_title('Spread de Preços')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 4. Fase 3: Análise de Oportunidades de Arbitragem

In [ ]:
# Identifica pontos de arbitragem
if df is not None:
    # Arbitragem ocorre quando YES + NO < 1 (spread > 0)
    arbitrage_points = df[df['spread'] > 0].copy()
    
    print(f"Total de pontos: {len(df)}")
    print(f"Pontos com arbitragem (spread > 0): {len(arbitrage_points)}")
    print(f"Percentual em arbitragem: {len(arbitrage_points)/len(df)*100:.1f}%")
    
    if len(arbitrage_points) > 0:
        print(f"\nSpread médio em arbitragem: {arbitrage_points['spread'].mean()*100:.2f}%")
        print(f"Spread máximo: {arbitrage_points['spread'].max()*100:.2f}%")

In [ ]:
# Distribuição do spread
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histograma do spread
    axes[0].hist(df['spread'] * 100, bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0, color='red', linestyle='--', label='Equilíbrio')
    axes[0].set_xlabel('Spread (%)')
    axes[0].set_ylabel('Frequência')
    axes[0].set_title('Distribuição do Spread')
    axes[0].legend()
    
    # Box plot do spread por dia da semana
    df['dayofweek'] = df['timestamp'].dt.day_name()
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df['dayofweek'] = pd.Categorical(df['dayofweek'], categories=day_order, ordered=True)
    df.boxplot(column='spread', by='dayofweek', ax=axes[1])
    axes[1].set_xlabel('Dia da Semana')
    axes[1].set_ylabel('Spread')
    axes[1].set_title('Spread por Dia da Semana')
    plt.suptitle('')  # Remove título automático
    
    plt.tight_layout()
    plt.show()

## 5. Fase 4: Estatísticas Agregadas

In [ ]:
def calculate_market_stats(df):
    """Calcula estatísticas de um mercado."""
    if df is None or len(df) == 0:
        return None
    
    return {
        'total_points': len(df),
        'spread_mean': df['spread'].mean(),
        'spread_std': df['spread'].std(),
        'spread_min': df['spread'].min(),
        'spread_max': df['spread'].max(),
        'arbitrage_count': len(df[df['spread'] > 0]),
        'arbitrage_pct': len(df[df['spread'] > 0]) / len(df) * 100,
        'avg_arb_spread': df[df['spread'] > 0]['spread'].mean() if len(df[df['spread'] > 0]) > 0 else 0,
    }

if df is not None:
    stats = calculate_market_stats(df)
    print("Estatísticas do Mercado:")
    for key, value in stats.items():
        if 'pct' in key or 'spread' in key:
            print(f"  {key}: {value*100 if 'pct' not in key else value:.2f}%")
        else:
            print(f"  {key}: {value}")

## 6. Análise por Hora do Dia

In [ ]:
# Análise de padrões por hora
if df is not None:
    df['hour'] = df['timestamp'].dt.hour
    hourly_stats = df.groupby('hour').agg({
        'spread': ['mean', 'std', 'count'],
        'price_yes': 'mean',
        'price_no': 'mean',
    }).round(4)
    
    print("Estatísticas por Hora:")
    display(hourly_stats)

In [ ]:
# Visualização por hora
if df is not None:
    hourly_spread = df.groupby('hour')['spread'].mean() * 100
    
    plt.figure(figsize=(12, 5))
    plt.bar(hourly_spread.index, hourly_spread.values, color='steelblue', edgecolor='black')
    plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    plt.xlabel('Hora do Dia (UTC)')
    plt.ylabel('Spread Médio (%)')
    plt.title('Spread Médio por Hora do Dia')
    plt.xticks(range(24))
    plt.tight_layout()
    plt.show()

## 7. Próximos Passos

Com a análise de preços e spreads concluída, podemos avançar para:

1. **Fase 5**: Comparação entre mercados
2. **Fase 6**: Análise temporal detalhada
3. **Fase 7**: Modelo de custos
4. **Fase 8**: Validação de edge
5. **Fase 9**: Framework de risco